In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re
import seaborn as sns
from scipy.stats import linregress
from pathlib import Path
import os


In [ ]:
DML_DIR = Path("../../results/figure3/data")
FIG_DIR = Path("../../results/figure3/figures_dml")
COEF_MAP_DIR = FIG_DIR / "coef_maps"
DIAG_DIR = Path("../../results/figure3/diagnostics")
FP_COEF = DML_DIR / "all_pixels_dml_TPRSOS.csv"

coef = pd.read_csv(FP_COEF)
required_p = ["p_T", "p_P", "p_R", "p_SOS"]
missing = [c for c in required_p if c not in coef.columns]
if missing:
    raise FileNotFoundError(
        f"Missing p-value columns {missing} in {FP_COEF}. "
        "Re-run figures/main/figure3_compute_dml.ipynb first."
    )
print("Loaded", FP_COEF)
print("n =", len(coef))
print(coef["region"].value_counts())
coef.head()


In [ ]:
P_THRESH_M = 1.0
MAT_SPLIT = 7.25

def _classify_region(t, p):
    p_m = float(p) if float(p) < 20 else float(p) / 1000.0
    if p_m >= P_THRESH_M:
        return "wet"
    return "hot-dry" if float(t) > MAT_SPLIT else "cold-dry"

coef = coef.copy()
coef["region"] = [
    _classify_region(t, p) for t, p in zip(coef["annual_t"], coef["annual_p"])
]
print("Reclassified regions (P <", P_THRESH_M, "m = dry):")
print(coef["region"].value_counts())

coef_hot_dry = coef[coef["region"] == "hot-dry"].copy()
coef_cold_dry = coef[coef["region"] == "cold-dry"].copy()
coef_wet = coef[coef["region"] == "wet"].copy()


## ATE vs MAT


In [ ]:
from matplotlib.ticker import MultipleLocator
from matplotlib.patches import Patch

veg_fp = "../../data/veg_class_data/tables/veg_class.csv"
if os.path.exists(veg_fp) and "veg_class" not in coef.columns:
    veg_class = pd.read_csv(veg_fp)
    coef = coef.merge(
        veg_class[["latitude", "longitude", "veg_class"]],
        on=["latitude", "longitude"], how="inner",
    )
df_plot = (
    coef[coef["veg_class"].isin([12.0, 13.0, 14.0])].copy().reset_index(drop=True)
    if "veg_class" in coef.columns else coef.copy()
)
prcp_mm = df_plot["annual_p"] * 1000 if df_plot["annual_p"].max() < 10 else df_plot["annual_p"]
mat = df_plot["annual_t"].to_numpy(float)
dry = (prcp_mm < 1000).to_numpy()
wet = ~dry

COLD_HOT_MAT_THRESH = 7.25  # cold-dry vs hot-dry
cold_dry_mask = dry & (mat < COLD_HOT_MAT_THRESH)
hot_dry_mask = dry & (mat >= COLD_HOT_MAT_THRESH)
wet_mask = wet

df_3c = df_plot.copy()
df_3c["b_absT_minus_absP"] = df_3c["b_T"].abs() - df_3c["b_P"].abs()
_absP = df_3c["b_P"].abs().to_numpy(float)
df_3c["b_absT_over_absP"] = np.where(_absP >= 1e-6, df_3c["b_T"].abs().to_numpy(float) / _absP, np.nan)

def bin_to_three_degree(vals, anchor=COLD_HOT_MAT_THRESH):
        vals = np.asarray(vals, dtype=float)
    left = anchor + np.floor((vals - anchor) / 3.0) * 3.0
    return left + 1.5

def binned_stats_3c(x_vals, z_vals, bins, x_min, x_max, min_n=20):
    means, twose, mids = [], [], []
    for xm in bins:
        if xm < x_min or xm > x_max:
            continue
        mask = (x_vals == xm) & np.isfinite(z_vals)
        n = int(np.sum(mask))
        if n >= min_n:
            m = float(np.nanmean(z_vals[mask]))
            s = float(np.nanstd(z_vals[mask]))
            means.append(m)
            twose.append(2.0 * s / np.sqrt(n))
            mids.append(float(xm))
    return np.asarray(mids), np.asarray(means), np.asarray(twose)

cd_lim1, cd_lim2 = -6.5, 7.25   # cold-dry midpoints are < 7.25
hd_lim1, hd_lim2 = 7.25, 14.0   # hot-dry midpoints are >= 8.75
w_lim1, w_lim2 = -4.5, 20.0
XLIM_3C = (-7.0, 20.0)

x_all_3c = bin_to_three_degree(mat)
bin_mids_3c = np.unique(np.sort(x_all_3c))
print("3°C bin midpoints (anchored at 7.25):", bin_mids_3c)
print("n cold-dry / hot-dry / wet:",
      int(cold_dry_mask.sum()), int(hot_dry_mask.sum()), int(wet_mask.sum()))

COLD_DRY_COLOR = "#3B83C4"
HOT_DRY_COLOR = "#e03c31"
WET_COLOR = "gray"
BAR_ALPHA = 0.8  # 20% transparent
BAR_W_3C = 1.0
FS_LABEL = 13
FS_TICK = 11
FS_LEG = 11
FIGSIZE_3C = (5.2, 3.6)

DIAG_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

legend_handles_3c = [
    Patch(facecolor=COLD_DRY_COLOR, edgecolor="none", alpha=BAR_ALPHA, label="Cold-dry regions"),
    Patch(facecolor=HOT_DRY_COLOR, edgecolor="none", alpha=BAR_ALPHA, label="Hot-dry regions"),
    Patch(facecolor=WET_COLOR, edgecolor="none", alpha=BAR_ALPHA, label="Wet regions"),
]

def _style_axes_xy_3c(ax, ylabel, y_step=0.2, xticks=None):
    ax.axhline(0, color="black", lw=0.7, zorder=0)
    ax.set_xlabel("MAT (°C)", fontsize=FS_LABEL)
    ax.set_ylabel(ylabel, fontsize=FS_LABEL)
    ax.tick_params(labelsize=FS_TICK)
    ax.yaxis.set_major_locator(MultipleLocator(y_step))
    if xticks is not None and len(xticks):
        xticks = np.asarray(xticks, dtype=float)
        pad = max(1.5, 0.6 * BAR_W_3C)
        xmin = float(xticks.min()) - pad
        xmax = float(xticks.max()) + pad
        ax.set_xlim(xmin, xmax)
        t0 = int(np.ceil(xmin / 6.0) * 6)
        t1 = int(np.floor(xmax / 6.0) * 6)
        tick_vals = np.arange(t0, t1 + 1, 6) if t1 >= t0 else np.array([])
        ax.set_xticks(tick_vals)
        ax.set_xticklabels([f"{int(t)}" for t in tick_vals])
    else:
        ax.set_xlim(XLIM_3C)
        ax.xaxis.set_major_locator(MultipleLocator(6))
    for sp in ("top", "right", "bottom", "left"):
        ax.spines[sp].set_visible(True)

def _auto_ylim_3c(*series):
    vals = []
    for mids, mean, se2 in series:
        if len(mids):
            vals.extend((mean - se2).tolist())
            vals.extend((mean + se2).tolist())
    vals = np.asarray(vals, dtype=float)
    vals = vals[np.isfinite(vals)]
    if len(vals) == 0:
        return -0.2, 0.4
    y0, y1 = float(vals.min()), float(vals.max())
    pad = 0.05 * max(y1 - y0, 0.1)
    ymin, ymax = y0 - pad, y1 + pad
    if ymin > 0:
        ymin = min(0.0, ymin)
    if ymax < 0:
        ymax = max(0.0, ymax)
    return ymin, ymax

def binned_absT_over_absP_3c(x_vals, t_vals, p_vals, bins, x_min, x_max, min_n=20):
        means, twose, mids = [], [], []
    for xm in bins:
        if xm < x_min or xm > x_max:
            continue
        mask = (x_vals == xm) & np.isfinite(t_vals) & np.isfinite(p_vals)
        n = int(np.sum(mask))
        if n < min_n:
            continue
        at = np.abs(t_vals[mask])
        ap = np.abs(p_vals[mask])
        mt, mp = float(np.mean(at)), float(np.mean(ap))
        if mp < 1e-6:
            continue
        ratio = mt / mp
        se_t = float(np.std(at, ddof=1)) / np.sqrt(n) if n > 1 else 0.0
        se_p = float(np.std(ap, ddof=1)) / np.sqrt(n) if n > 1 else 0.0
        se_r = ratio * np.sqrt((se_t / max(mt, 1e-12)) ** 2 + (se_p / mp) ** 2)
        means.append(ratio)
        twose.append(2.0 * se_r)
        mids.append(float(xm))
    return np.asarray(mids), np.asarray(means), np.asarray(twose)

def _mat_series_3c(vkey):
    if vkey == "b_absT_over_absP":
        t = df_3c["b_T"].to_numpy(float)
        p = df_3c["b_P"].to_numpy(float)
        mids_cd, mean_cd, se2_cd = binned_absT_over_absP_3c(
            x_all_3c[cold_dry_mask], t[cold_dry_mask], p[cold_dry_mask], bin_mids_3c, cd_lim1, cd_lim2
        )
        mids_hd, mean_hd, se2_hd = binned_absT_over_absP_3c(
            x_all_3c[hot_dry_mask], t[hot_dry_mask], p[hot_dry_mask], bin_mids_3c, hd_lim1, hd_lim2
        )
        mids_w, mean_w, se2_w = binned_absT_over_absP_3c(
            x_all_3c[wet_mask], t[wet_mask], p[wet_mask], bin_mids_3c, w_lim1, w_lim2
        )
        return (mids_cd, mean_cd, se2_cd, mids_hd, mean_hd, se2_hd, mids_w, mean_w, se2_w)
    z = df_3c[vkey].to_numpy(float)
    mids_cd, mean_cd, se2_cd = binned_stats_3c(
        x_all_3c[cold_dry_mask], z[cold_dry_mask], bin_mids_3c, cd_lim1, cd_lim2
    )
    mids_hd, mean_hd, se2_hd = binned_stats_3c(
        x_all_3c[hot_dry_mask], z[hot_dry_mask], bin_mids_3c, hd_lim1, hd_lim2
    )
    mids_w, mean_w, se2_w = binned_stats_3c(
        x_all_3c[wet_mask], z[wet_mask], bin_mids_3c, w_lim1, w_lim2
    )
    return (mids_cd, mean_cd, se2_cd, mids_hd, mean_hd, se2_hd, mids_w, mean_w, se2_w)

def _apply_ylim_3c(ax, ylabel, ylim_fixed, series, stub, floor0=False, xticks=None):
    if ylim_fixed is None:
        ymin, ymax = _auto_ylim_3c(*series)
        if floor0:
            ymin = min(0.0, ymin) if ymin < 0 else 0.0
        yr = ymax - ymin
        y_step = 0.1 if yr <= 0.6 else (0.2 if yr <= 2.0 else 0.5)
        _style_axes_xy_3c(ax, ylabel, y_step=y_step, xticks=xticks)
        ax.set_ylim(ymin, ymax)
        print(f"{stub} ylim = ({ymin:.3f}, {ymax:.3f})")
    else:
        _style_axes_xy_3c(ax, ylabel, y_step=0.2, xticks=xticks)
        ax.set_ylim(*ylim_fixed)

def plot_mat_bars_3c(vkey, stub_diag, stub_fig, ylabel, ylim_fixed=None, show_legend=False):
    mids_cd, mean_cd, se2_cd, mids_hd, mean_hd, se2_hd, mids_w, mean_w, se2_w = _mat_series_3c(vkey)

    fig, ax = plt.subplots(figsize=FIGSIZE_3C, dpi=500)
    if len(mids_cd):
        ax.bar(
            mids_cd - BAR_W_3C / 2, mean_cd, width=BAR_W_3C, yerr=se2_cd,
            color=COLD_DRY_COLOR, alpha=BAR_ALPHA, ecolor="0.3", capsize=1.0,
            zorder=2, error_kw=dict(lw=0.8), align="center",
        )
    if len(mids_hd):
        ax.bar(
            mids_hd - BAR_W_3C / 2, mean_hd, width=BAR_W_3C, yerr=se2_hd,
            color=HOT_DRY_COLOR, alpha=BAR_ALPHA, ecolor="0.3", capsize=1.0,
            zorder=2, error_kw=dict(lw=0.8), align="center",
        )
    if len(mids_w):
        ax.bar(
            mids_w + BAR_W_3C / 2, mean_w, width=BAR_W_3C, yerr=se2_w,
            color=WET_COLOR, alpha=BAR_ALPHA, ecolor="0.3", capsize=1.0,
            zorder=2, error_kw=dict(lw=0.8), align="center",
        )

    series = [
        (mids_cd, mean_cd, se2_cd),
        (mids_hd, mean_hd, se2_hd),
        (mids_w, mean_w, se2_w),
    ]
    xticks = np.unique(np.concatenate(
        [m for m in (mids_cd, mids_hd, mids_w) if len(m)]
    )) if any(len(m) for m in (mids_cd, mids_hd, mids_w)) else np.array([])
    _apply_ylim_3c(
        ax, ylabel, ylim_fixed, series, stub_fig,
        floor0=(vkey == "b_absT_over_absP"), xticks=xticks,
    )
    if vkey == "b_absT_over_absP":
        ax.axhline(1.0, color="black", lw=0.7, zorder=0)
    if show_legend:
        ax.legend(handles=legend_handles_3c, frameon=False, fontsize=FS_LEG, loc="upper left")
    fig.tight_layout()
    fp_diag = DIAG_DIR / stub_diag
    fp_fig = FIG_DIR / stub_fig
    fig.savefig(fp_diag, dpi=500, bbox_inches="tight")
    fig.savefig(fp_fig, dpi=500, bbox_inches="tight")
    print("Saved", fp_diag)
    print("Saved", fp_fig)
    print(f"  cold-dry mids={mids_cd.tolist()}  hot-dry mids={mids_hd.tolist()}")
    plt.show()
    return fig

plot_mat_bars_3c(
    "b_T",
    "dml_T_MAT_3C_bars.png",
    "fig3_dml_MAT_T_3C_bars.png",
    "Effect size (T)",
    ylim_fixed=(-0.2, 0.4),
    show_legend=True,
)
plot_mat_bars_3c(
    "b_P",
    "dml_P_MAT_3C_bars.png",
    "fig3_dml_MAT_P_3C_bars.png",
    "Effect size (P)",
    ylim_fixed=(-0.2, 0.4),
)
plot_mat_bars_3c(
    "b_absT_minus_absP",
    "dml_absTminusAbsP_MAT_3C_bars.png",
    "fig3_dml_absTminusAbsP_MAT_3C_bars.png",
    "Effect size (|T|−|P|)",
    ylim_fixed=None,
)
plot_mat_bars_3c(
    "b_absT_over_absP",
    "dml_absToverAbsP_MAT_3C_bars.png",
    "fig3_dml_absToverAbsP_MAT_3C_bars.png",
    r"Effect size (|T|/|P|)",
    ylim_fixed=None,
)


In [ ]:
FIG_DIR.mkdir(parents=True, exist_ok=True)

def plot_dml_ates_grouped(region_dfs, coef_names, out_path, ylim, figsize=(5.4, 3.8), ylabel=None):
    from matplotlib.patches import Patch
    from matplotlib.ticker import MultipleLocator
    from scipy.stats import ttest_ind

    label_map = {
        "b_T": "Preseason T",
        "b_P": "Preseason P",
        "b_R": "Preseason R",
        "b_SOS": "SOS",
    }
    region_colors = {
        "cold-dry": "#3B83C4",  # keep evaluation SIAM blue
        "hot-dry": "#e03c31",   # Fig. 1 dry
        "wet": "gray",          # Fig. 1 wet
    }
    regions = [
        ("cold-dry", region_dfs["cold-dry"]),
        ("hot-dry", region_dfs["hot-dry"]),
        ("wet", region_dfs["wet"]),
    ]
    reg_short = ["CD", "HD", "W"]

    def mean_and_se(series):
        values = series.dropna()
        if len(values) == 0:
            return np.nan, np.nan
        if len(values) == 1:
            return float(values.iloc[0]), np.nan
        return float(values.mean()), float(values.std(ddof=1) / np.sqrt(len(values)))

    def stars(p):
        if p < 0.01:
            return "**"
        if p < 0.05:
            return "*"
        return "ns"

    group_gap = 0.68
    x = np.arange(len(coef_names)) * group_gap
    n_reg = len(regions)
    width = 0.15
    offsets = (np.arange(n_reg) - (n_reg - 1) / 2.0) * width

    fig, ax = plt.subplots(figsize=figsize)
    y_lo0, y_hi0 = ylim
    y_span0 = y_hi0 - y_lo0

    positions = {}
    series = {}
    label_y = {}  # where the mean number sits

    for j, (reg_name, df) in enumerate(regions):
        means, ses = [], []
        for cname in coef_names:
            m, se = mean_and_se(df[cname])
            means.append(m)
            ses.append(0.0 if np.isnan(se) else se)
        color = region_colors[reg_name]
        bars = ax.bar(
            x + offsets[j], means, width=width * 0.95, color=color,
            alpha=0.8,  # 90% transparent
            yerr=[2 * np.array(ses)], capsize=3,
            edgecolor="none", linewidth=0,
            error_kw=dict(ecolor="#333333", lw=0.8, capthick=0.8),
        )
        for i, (bar, val, se) in enumerate(zip(bars, means, ses)):
            series[(i, j)] = df[coef_names[i]].dropna().to_numpy()
            pos = float(bar.get_x() + bar.get_width() / 2.0)
            positions[(i, j)] = pos
            if np.isnan(val):
                continue
            label = "0.00" if abs(val) < 5e-3 else f"{val:.2f}"
            num_pad = 0.018 * y_span0
            if val >= 0:
                ypos = val + 2 * se + num_pad
                va = "bottom"
            else:
                ypos = val - 2 * se - num_pad
                va = "top"
            label_y[(i, j)] = ypos
            ax.text(
                pos, ypos, label,
                ha="center", va=va, fontsize=10.5, color="#222222",
            )

    top_pairs = [(0, 1), (1, 2)]
    bot_pairs = [(0, 2)]
    step = 0.07 * y_span0
    tick = 0.015 * y_span0
    gap = 0.16 * y_span0  # room for number glyph height

    y_min, y_max = y_lo0, y_hi0

    for i, cname in enumerate(coef_names):
        hi = 0.0
        lo = 0.0
        for j in range(n_reg):
            if (i, j) not in label_y:
                continue
            m, se = mean_and_se(regions[j][1][cname])
            se = 0.0 if np.isnan(se) else se
            if m >= 0:
                hi = max(hi, label_y[(i, j)], m + 2 * se)
                lo = min(lo, 0.0)
            else:
                lo = min(lo, label_y[(i, j)], m - 2 * se)
                hi = max(hi, 0.0)

        base_top = hi + gap
        base_bot = lo - gap

        for k, (j1, j2) in enumerate(top_pairs):
            p = ttest_ind(series[(i, j1)], series[(i, j2)], equal_var=False, nan_policy="omit").pvalue
            s = stars(p)
            x1, x2 = positions[(i, j1)], positions[(i, j2)]
            y = base_top + k * step
            ax.plot([x1, x1, x2, x2], [y - tick, y, y, y - tick],
                    color="#333333", lw=0.7, clip_on=False)
            ax.text(0.5 * (x1 + x2), y + tick * 0.25, s,
                    ha="center", va="bottom", fontsize=8, color="#222222")
            print(f"{cname} {reg_short[j1]} vs {reg_short[j2]}: p={p:.2e} → {s}")
            y_max = max(y_max, y + 1.2 * step)

        for k, (j1, j2) in enumerate(bot_pairs):
            p = ttest_ind(series[(i, j1)], series[(i, j2)], equal_var=False, nan_policy="omit").pvalue
            s = stars(p)
            x1, x2 = positions[(i, j1)], positions[(i, j2)]
            y = base_bot - k * step
            ax.plot([x1, x1, x2, x2], [y + tick, y, y, y + tick],
                    color="#333333", lw=0.7, clip_on=False)
            ax.text(0.5 * (x1 + x2), y - tick * 0.25, s,
                    ha="center", va="top", fontsize=8, color="#222222")
            print(f"{cname} {reg_short[j1]} vs {reg_short[j2]} (below): p={p:.2e} → {s}")
            y_min = min(y_min, y - 1.2 * step)

    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(
        [label_map[c] for c in coef_names],
        fontsize=10.5, rotation=45, ha="right", rotation_mode="anchor",
    )
    ylab_map = {
        "b_T": "Effect size (T)",
        "b_P": "Effect size (P)",
        "b_R": "Effect size (R)",
        "b_SOS": "Effect size (SOS)",
    }
    if ylabel is None:
        ylabel = ylab_map[coef_names[0]] if len(coef_names) == 1 else None
        if ylabel is None:
            raise ValueError("Pass ylabel= or plot one coef at a time (avoid mixed T/P y-title)")
    ax.set_ylabel(ylabel, fontsize=11)
    ax.tick_params(axis="y", labelsize=10)
    ax.set_ylim(y_min, y_max)
    ax.yaxis.set_major_locator(MultipleLocator(0.1))
    ax.set_xlim(x[0] - 0.38, x[-1] + 0.38)

    handles = [
        Patch(facecolor=region_colors["cold-dry"], edgecolor="none", alpha=0.8, label="Cold-dry regions"),
        Patch(facecolor=region_colors["hot-dry"], edgecolor="none", alpha=0.8, label="Hot-dry regions"),
        Patch(facecolor=region_colors["wet"], edgecolor="none", alpha=0.8, label="Wet regions"),
    ]
    ax.legend(
        handles=handles,
        frameon=False,
        fontsize=7.5,
        loc="upper right",
        handlelength=2.4,
        handleheight=0.75,
        handletextpad=0.35,
        labelspacing=0.25,
        borderaxespad=0.2,
    )

    fig.tight_layout()
    for sp in ("top", "bottom", "left", "right"):
        ax.spines[sp].set_visible(True)
        ax.spines[sp].set_color("black")
        ax.spines[sp].set_linewidth(0.5)
    fig.savefig(out_path, dpi=500, bbox_inches="tight")
    plt.show()
    return fig

def plot_dml_ates(dfs, names):
        region_dfs = {
        "cold-dry": dfs[0],
        "hot-dry": dfs[1],
        "wet": dfs[2],
    }
    figs = []
    for cname, stub, ylim, figsize in [
        ("b_T", "fig3_dml_1a_T.png", (-0.10, 0.25), (3.2, 3.8)),
        ("b_P", "fig3_dml_1a_P.png", (-0.10, 0.25), (3.2, 3.8)),
        ("b_R", "fig3_dml_1b_R.png", (-0.35, 0.15), (3.2, 3.8)),
        ("b_SOS", "fig3_dml_1b_SOS.png", (-0.35, 0.15), (3.2, 3.8)),
    ]:
        figs.append(plot_dml_ates_grouped(
            region_dfs, [cname], str(FIG_DIR / stub),
            ylim=ylim, figsize=figsize,
        ))
    return tuple(figs)


## Regional means


In [ ]:
region_order = [coef_cold_dry, coef_hot_dry, coef_wet]
figs = plot_dml_ates(
    region_order,
    ["Cold-dry regions", "Hot-dry regions", "Wet regions"]
)
fig = figs[0]


## Moving window


In [ ]:
from matplotlib.ticker import MultipleLocator
from scipy.stats import linregress

FP_MW = DML_DIR / "moving_window_20y_dml_T_P_TxP.csv"  # legacy file; plot T & P only
FP_TREND = DML_DIR / "moving_window_20y_dml_T_P_trends.csv"
mw = pd.read_csv(FP_MW)

var_specs = [
    ("b_T", "Preseason T", "#c14d48"),
    ("b_P", "Preseason P", "#4d91c4"),
]
region_styles = {
    "cold-dry": dict(ls="-", marker="o"),
    "hot-dry": dict(ls="--", marker="s"),
    "wet": dict(ls=":", marker="^"),
}
region_order = ["cold-dry", "hot-dry", "wet"]
region_labels = {
    "cold-dry": "Cold-dry",
    "hot-dry": "Hot-dry",
    "wet": "Wet",
}

def _stars(p):
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return "ns"

trend_rows = []
fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.4), sharey=True)
for ax, (vkey, title, color) in zip(axes, var_specs):
    y_annot = 0.98
    for reg in region_order:
        sub = mw[mw["region"] == reg].sort_values("year_center")
        y = sub[f"{vkey}_mean"].to_numpy(float)
        se = sub[f"{vkey}_se"].to_numpy(float)
        x = sub["year_center"].to_numpy(float)
        st = region_styles[reg]
        ax.plot(
            x, y, color=color, lw=1.4,
            ls=st["ls"], marker=st["marker"], ms=4,
            label=region_labels[reg],
        )
        ax.fill_between(x, y - se, y + se, color=color, alpha=0.15, linewidth=0)

        m = np.isfinite(x) & np.isfinite(y)
        res = linregress(x[m], y[m])
        xx = np.array([x[m].min(), x[m].max()])
        ax.plot(xx, res.intercept + res.slope * xx, color=color, lw=1.0,
                ls=st["ls"], alpha=0.55)
        sig = _stars(res.pvalue)
        trend_rows.append({
            "region": reg, "variable": vkey,
            "slope_per_year": float(res.slope),
            "slope_per_decade": float(res.slope) * 10,
            "r_value": float(res.rvalue),
            "r2": float(res.rvalue ** 2),
            "p_value": float(res.pvalue),
            "stderr": float(res.stderr),
            "n_windows": int(m.sum()),
            "sig": sig,
        })
        ax.text(
            0.02, y_annot,
            f"{region_labels[reg]}: {res.slope*10:+.3f}/dec {sig}",
            transform=ax.transAxes, fontsize=7.5, color=color,
            va="top", ha="left",
        )
        y_annot -= 0.10
        print(f"{title:18s} {reg:9s}  {res.slope*10:+.4f}/dec  p={res.pvalue:.3e} {sig}")

    ax.axhline(0, color="black", lw=0.7)
    ax.set_xlabel("Central year of moving window", fontsize=10)
    ax.tick_params(labelsize=9)
    ax.set_ylim(-0.1, 0.37)
    ax.yaxis.set_major_locator(MultipleLocator(0.1))
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)

for ax, (vkey, _title, _color) in zip(axes, var_specs):
    short = {"b_T": "T", "b_P": "P"}.get(vkey, vkey)
    ax.set_ylabel(f"Effect size ({short})", fontsize=11)
axes[0].legend(frameon=False, fontsize=8, loc="lower right")
fig.tight_layout()
FIG_DIR.mkdir(parents=True, exist_ok=True)
out = FIG_DIR / "fig3_dml_moving_window_20y_T_P.png"
fig.savefig(out, dpi=500, bbox_inches="tight")
print("Saved", out)

tr = pd.DataFrame(trend_rows)
tr.to_csv(FP_TREND, index=False)
print("Saved trends →", FP_TREND)
print(tr[["region", "variable", "slope_per_decade", "p_value", "sig"]].to_string(index=False))
plt.show()


## Maps


In [ ]:
import rasterio as rs
from rasterio.features import rasterize
import geopandas as gpd
from shapely.geometry import box
import cartopy.crs as ccrs
from matplotlib.colors import TwoSlopeNorm
import matplotlib.path as mpath
import matplotlib.ticker as mticker
from matplotlib.patches import Patch
import cartopy.io.shapereader as shpreader

COEF_SPECS = [
    ("b_T", "p_T", "T"),
    ("b_P", "p_P", "P"),
    ("b_R", "p_R", "R"),
    ("b_SOS", "p_SOS", "SOS"),
]

POS_LIGHT, POS_DEEP = "#e67a73", "#bf1216"
NEG_LIGHT, NEG_DEEP = "#6cbfd9", "#1362bf"

def _base_grid():
    with rs.open("../../data/satellite_data/images/base-image/test.tif") as src:
        return src.transform, src.width, src.height, src.crs

def _land_mask(transform, width, height, crs):
    ne_path = shpreader.natural_earth(resolution="110m", category="physical", name="land")
    land = gpd.read_file(ne_path).to_crs(crs)
    left, bottom, right, top = rs.transform.array_bounds(height, width, transform)
    land = gpd.clip(land, gpd.GeoDataFrame(geometry=[box(left, bottom, right, top)], crs=crs))
    return rasterize(
        [(geom, 1) for geom in land.geometry],
        out_shape=(height, width),
        transform=transform,
        fill=0,
        dtype="uint8",
    )

def rasterize_values(gdf, value_col, transform, width, height, land_mask):
    shapes = ((geom, float(val)) for geom, val in zip(gdf.geometry, gdf[value_col]) if np.isfinite(val))
    rast = rasterize(
        shapes=shapes,
        out_shape=(height, width),
        transform=transform,
        fill=np.nan,
        dtype="float32",
    )
    return np.where(land_mask == 1, rast, np.nan)

def _polar_ax(fig, nrows, ncols, index):
    from cartopy.mpl.ticker import LongitudeFormatter

    ax = fig.add_subplot(nrows, ncols, index, projection=ccrs.NorthPolarStereo())
    ax.set_extent([-180, 180, 30, 90], ccrs.PlateCarree())
    ax.coastlines(linewidth=0.4)
    theta = np.linspace(0, 2 * np.pi, 100)
    verts = np.vstack([np.sin(theta), np.cos(theta)]).T
    ax.set_boundary(mpath.Path(verts * 0.5 + [0.5, 0.5]), transform=ax.transAxes)
    gl = ax.gridlines(linewidth=0.4, color="gray", alpha=0.6, linestyle="--")
    gl.xlocator = mticker.FixedLocator(np.arange(-180, 181, 60))
    gl.ylocator = mticker.FixedLocator([30, 50, 70])

    lon_formatter = LongitudeFormatter()
    for lon in np.arange(-180, 181, 60):
        if lon in (-180, -60):
            continue
        label_lon, label_lat = lon, 25
        if lon in (-120, 120):
            label_lat = 23
        if lon == 0:
            label_lon, label_lat = 2, 29
        if lon == 180:
            label_lon = 178
        ax.text(
            label_lon, label_lat, lon_formatter(lon),
            transform=ccrs.PlateCarree(), ha="center", va="top",
            fontsize=8, color="black",
        )
    return ax

def _inset_sign_bar(ax_bar, values, pvals, alpha=0.05):
    from matplotlib.ticker import FixedLocator, FixedFormatter

    v = np.asarray(values, float)
    p = np.asarray(pvals, float)
    m = np.isfinite(v) & np.isfinite(p)
    v, p = v[m], p[m]
    n = len(v)
    if n == 0:
        return {}
    neg_sig = np.sum((v < 0) & (p < alpha)) / n * 100
    neg_ns = np.sum((v < 0) & (p >= alpha)) / n * 100
    pos_ns = np.sum((v > 0) & (p >= alpha)) / n * 100
    pos_sig = np.sum((v > 0) & (p < alpha)) / n * 100

    ax_bar.bar(0, pos_sig, width=0.75, color=POS_DEEP, edgecolor="none", linewidth=0)
    ax_bar.bar(0, pos_ns, width=0.75, bottom=pos_sig, color=POS_LIGHT, edgecolor="none", linewidth=0)
    ax_bar.bar(1, neg_sig, width=0.75, color=NEG_DEEP, edgecolor="none", linewidth=0)
    ax_bar.bar(1, neg_ns, width=0.75, bottom=neg_sig, color=NEG_LIGHT, edgecolor="none", linewidth=0)

    ax_bar.set_xlim(-0.7, 1.7)
    ax_bar.set_ylim(0, 100)
    ax_bar.yaxis.set_major_locator(FixedLocator([0, 30, 60, 90]))
    ax_bar.yaxis.set_major_formatter(FixedFormatter(["0", "30", "60", "90"]))
    ax_bar.set_xticks([0, 1])
    ax_bar.set_xticklabels(["+", "−"], fontsize=13)
    ax_bar.set_ylabel("Percentage (%)", fontsize=13)
    ax_bar.tick_params(axis="y", labelsize=12, length=5, width=1.2, colors="black")
    ax_bar.tick_params(axis="x", labelsize=13, length=4, width=1.2, colors="black", pad=2)
    ax_bar.spines["top"].set_visible(False)
    ax_bar.spines["right"].set_visible(False)
    ax_bar.spines["left"].set_visible(True)
    ax_bar.spines["bottom"].set_visible(True)
    ax_bar.spines["left"].set_linewidth(1.4)
    ax_bar.spines["bottom"].set_linewidth(1.4)
    ax_bar.spines["left"].set_color("black")
    ax_bar.spines["bottom"].set_color("black")
    ax_bar.set_facecolor("white")
    ax_bar.patch.set_alpha(1.0)
    ax_bar.patch.set_visible(True)
    return dict(neg_sig=neg_sig, neg_ns=neg_ns, pos_ns=pos_ns, pos_sig=pos_sig)

def show_coefficient_maps(df, out_dir=None):
    if out_dir is None:
        out_dir = COEF_MAP_DIR
    from pathlib import Path
    from cartopy.mpl.ticker import LongitudeFormatter
    from matplotlib.cm import ScalarMappable

    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    transform, width, height, crs = _base_grid()
    land_mask = _land_mask(transform, width, height, crs)
    gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.longitude, df.latitude), crs=crs)
    left0, bottom0, right0, top0 = rs.transform.array_bounds(height, width, transform)
    gdf = gdf[gdf.geometry.within(box(left0, bottom0, right0, top0))].copy()
    print(f"[DEBUG] Points for coef maps: {len(gdf)}")

    all_vals = np.concatenate([gdf[b].to_numpy(float) for b, _, _ in COEF_SPECS])
    all_vals = all_vals[np.isfinite(all_vals)]
    vmax = float(np.nanpercentile(np.abs(all_vals), 95)) if len(all_vals) else 0.3
    vmax = max(vmax, 0.05)
    vmax = 0.45
    norm = TwoSlopeNorm(vmin=-0.45, vcenter=0.0, vmax=0.45)
    print(f"Universal color scale: ±{vmax:.3f}")

    file_stub = {"T": "T", "P": "P", "R": "R", "SOS": "SOS"}
    stats = {}
    figs = []

    for bcol, pcol, title in COEF_SPECS:
        fig = plt.figure(figsize=[5, 5], facecolor="white")
        ax = fig.add_subplot(1, 1, 1, projection=ccrs.NorthPolarStereo())
        ax.set_extent([-180, 180, 30, 90], ccrs.PlateCarree())
        ax.coastlines()  # same default as dominant-driver map

        theta = np.linspace(0, 2 * np.pi, 100)
        center, radius = [0.5, 0.5], 0.5
        verts = np.vstack([np.sin(theta), np.cos(theta)]).T
        ax.set_boundary(mpath.Path(verts * radius + center), transform=ax.transAxes)

        gl = ax.gridlines(linewidth=0.5, color="gray", alpha=0.7, linestyle="--")
        gl.xlocator = mticker.FixedLocator(np.arange(-180, 181, 60))
        gl.ylocator = mticker.FixedLocator([30, 50, 70])

        lon_formatter = LongitudeFormatter()
        for lon in np.arange(-180, 181, 60):
            if lon in (-180, -60):
                continue
            label_lon, label_lat = lon, 25
            if lon in (-120, 120):
                label_lat = 23
            if lon == 0:
                label_lon, label_lat = 2, 29
            if lon == 180:
                label_lon = 178
            ax.text(
                label_lon, label_lat, lon_formatter(lon),
                transform=ccrs.PlateCarree(), ha="center", va="top",
                fontsize=12, color="black",
            )

        rast = rasterize_values(gdf, bcol, transform, width, height, land_mask)
        left, bottom, right, top = rs.transform.array_bounds(height, width, transform)
        ax.imshow(
            np.ma.masked_invalid(rast),
            cmap="RdBu_r",
            norm=norm,
            extent=[left, right, bottom, top],
            transform=ccrs.PlateCarree(),
            origin="upper",
            interpolation="nearest",
        )

                ax_bar = fig.add_axes([0.12, 0.10, 0.24, 0.26], facecolor="white")
        stats[title] = _inset_sign_bar(ax_bar, gdf[bcol].to_numpy(), gdf[pcol].to_numpy())

        out_path = out_dir / f"fig3_coef_map_{file_stub[title]}.png"
        fig.savefig(out_path, dpi=500, bbox_inches="tight", facecolor="white", transparent=False)
        plt.show()
        figs.append(fig)
        print("Saved", out_path)

    fig_cb = plt.figure(figsize=(1.3, 3.8), facecolor="white")
    cax = fig_cb.add_axes([0.32, 0.08, 0.28, 0.84])
    sm = ScalarMappable(norm=norm, cmap="RdBu_r")
    sm.set_array([])
    cb = fig_cb.colorbar(sm, cax=cax)
    cb.set_ticks([-0.4, -0.2, 0.0, 0.2, 0.4])
    cb.set_ticklabels(["-0.4", "-0.2", "0", "0.2", "0.4"])
    cb.set_label("Effect size", fontsize=15)
    cb.ax.tick_params(labelsize=14, length=6, width=1.1)
    cb.outline.set_visible(False)
    cb_path = out_dir / "fig3_coef_colorbar.png"
    fig_cb.savefig(cb_path, dpi=400, bbox_inches="tight", facecolor="white", transparent=False)
    plt.show()
    print("Saved", cb_path)

    for k, v in stats.items():
        print(
            f"  {k}: neg={v['neg_sig']+v['neg_ns']:.1f}% (sig {v['neg_sig']:.1f}%), "
            f"pos={v['pos_sig']+v['pos_ns']:.1f}% (sig {v['pos_sig']:.1f}%)"
        )
    return figs


In [ ]:
figs_coef = show_coefficient_maps(coef)


## Dominant driver map


In [ ]:
import rasterio as rs
from rasterio.features import rasterize
import geopandas as gpd
from shapely.geometry import box
import cartopy.crs as ccrs
from matplotlib.colors import ListedColormap
import matplotlib.path as mpath

DRIVER_LABEL_MAP = {
    "b_T": "T",
    "b_P": "P",
    "b_R": "R",
    "b_SOS": "SOS",
}

DRIVER_COLORS = {
    "b_T": "#c14d48",
    "b_P": "#4d91c4",
    "b_R": "#7a5fa0",
    "b_SOS": "#ff69b4",
}

def assign_dominant_driver(df):
    use_cols = [c for c in DRIVER_LABEL_MAP if c in df.columns]
    df_combined = df.copy()
    abs_vals = df_combined[use_cols].abs()
    df_combined["dominant_driver"] = abs_vals.idxmax(axis=1)
    driver_codes = {name: i for i, name in enumerate(use_cols)}
    df_combined["dominant_driver_code"] = df_combined["dominant_driver"].map(driver_codes)
    return df_combined, driver_codes

def rasterize_driver(gdf, transform, width, height, crs):
    shapes = ((geom, value) for geom, value in zip(gdf.geometry, gdf["dominant_driver_code"]))
    driver_raster = rasterize(
        shapes=shapes,
        out_shape=(height, width),
        transform=transform,
        fill=np.nan,
        dtype="float32",
    )

    import cartopy.io.shapereader as shpreader

    ne_path = shpreader.natural_earth(resolution="110m", category="physical", name="land")
    land = gpd.read_file(ne_path).to_crs(crs)
    left, bottom, right, top = rs.transform.array_bounds(height, width, transform)
    raster_bounds = box(left, bottom, right, top)
    land = gpd.clip(land, gpd.GeoDataFrame(geometry=[raster_bounds], crs=crs))
    land_mask = rasterize(
        [(geom, 1) for geom in land.geometry],
        out_shape=(height, width),
        transform=transform,
        fill=0,
        dtype="uint8",
    )
    driver_raster = np.where(land_mask == 1, driver_raster, np.nan)
    print("[DEBUG] Pixels after land mask:", np.count_nonzero(~np.isnan(driver_raster)))
    return driver_raster

def show_driver_raster(driver_raster, transform, driver_colors, driver_labels):
    fig = plt.figure(figsize=[5, 5])
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.NorthPolarStereo())
    ax.set_extent([-180, 180, 30, 90], ccrs.PlateCarree())
    ax.coastlines()

    theta = np.linspace(0, 2 * np.pi, 100)
    center, radius = [0.5, 0.5], 0.5
    verts = np.vstack([np.sin(theta), np.cos(theta)]).T
    circle = mpath.Path(verts * radius + center)
    ax.set_boundary(circle, transform=ax.transAxes)

    cmap = ListedColormap(driver_colors)
    left, bottom, right, top = rs.transform.array_bounds(
        driver_raster.shape[0], driver_raster.shape[1], transform
    )
    driver_masked = np.ma.masked_invalid(driver_raster)

    ax.imshow(
        driver_masked,
        cmap=cmap,
        extent=[left, right, bottom, top],
        transform=ccrs.PlateCarree(),
        origin="upper",
        interpolation="nearest",
        vmin=0,
        vmax=len(driver_labels) - 1,
    )

    valid = driver_raster[~np.isnan(driver_raster)].astype(np.int64)
    counts = np.bincount(valid, minlength=len(driver_labels))
    percentages = counts / counts.sum() * 100 if counts.sum() > 0 else np.zeros(len(driver_labels))

    ax_bar = fig.add_axes([0.12, 0.10, 0.35, 0.25])
    ax_bar.bar(range(len(driver_labels)), percentages, color=driver_colors)
    pretty_labels = [DRIVER_LABEL_MAP.get(lbl, lbl) for lbl in driver_labels]
    ax_bar.set_xticks(range(len(driver_labels)))
    ax_bar.set_xticklabels(pretty_labels, rotation=45, ha="right", fontsize=7)
    ax_bar.set_ylabel("Percentage (%)", fontsize=8)
    ax_bar.set_ylim(0, max(percentages) * 1.2 if percentages.sum() > 0 else 1)
    ax_bar.tick_params(axis="y", labelsize=7)
    ax_bar.spines["top"].set_visible(False)
    ax_bar.spines["right"].set_visible(False)

    import matplotlib.ticker as mticker
    from cartopy.mpl.ticker import LongitudeFormatter

    gl = ax.gridlines(linewidth=0.5, color="gray", alpha=0.7, linestyle="--")
    gl.xlocator = mticker.FixedLocator(np.arange(-180, 181, 60))
    gl.ylocator = mticker.FixedLocator([30, 50, 70])
    lon_formatter = LongitudeFormatter()
    for lon in np.arange(-180, 181, 60):
        if lon in (-180, -60):
            continue
        label_lon, label_lat = lon, 25
        if lon in (-120, 120):
            label_lat = 23
        if lon == 0:
            label_lon, label_lat = 2, 29
        if lon == 180:
            label_lon = 178
        ax.text(
            label_lon, label_lat, lon_formatter(lon),
            transform=ccrs.PlateCarree(), ha="center", va="top", fontsize=12, color="black",
        )

    FIG_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIG_DIR / "fig3_2.png", dpi=500, bbox_inches="tight")
    plt.show()

def show_dominant_driver_map(df):
    with rs.open("../../data/satellite_data/images/base-image/test.tif") as src:
        transform = src.transform
        width, height = src.width, src.height
        crs = src.crs

    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df.longitude, df.latitude),
        crs=crs,
    )
    left, bottom, right, top = rs.transform.array_bounds(height, width, transform)
    raster_bounds = box(left, bottom, right, top)
    gdf = gdf[gdf.geometry.within(raster_bounds)]
    print(f"[DEBUG] Points inside raster bounds: {len(gdf)}")

    gdf, driver_codes = assign_dominant_driver(gdf)
    driver_raster = rasterize_driver(gdf, transform, width, height, crs)

    driver_labels = list(driver_codes.keys())
    driver_colors = [DRIVER_COLORS[lbl] for lbl in driver_labels]
    show_driver_raster(driver_raster, transform, driver_colors, driver_labels)


In [ ]:
show_dominant_driver_map(coef)
